In [2]:
import pandas as pd
import numpy as np

In [3]:
combined_df = pd.read_parquet("combined_raw.parquet")

If the cell is NaN, "", or whitespace, it becomes "NoFeedback". Everything else is mapped to Correct, Incorrect, or Error if there were error messages. Remaining values are assigned to "OtherFeedback".

outcome_raw is the cleaned copy of the original feedback_classification column.


In [4]:
def simplify_outcome(x):
    if pd.isna(x):
        return "NoFeedback"

    value = str(x).strip()
    if not value:
        return "NoFeedback"

    value_lower = value.lower()

    if value_lower in {"correct", "success"}:
        return "Correct"
    if value_lower in {"incorrect", "failure"}:
        return "Incorrect"
    if ("error" in value_lower) or ("exception" in value_lower):
        return "Error"

    return "OtherFeedback"

combined_df["outcome_raw"] = combined_df["feedback_classification"].astype("string").str.strip()
combined_df["outcome"] = combined_df["outcome_raw"].apply(simplify_outcome)


We notice a high proportion of "NoFeedBack" but that is most likely reflecting the rows in our datasets which aren't actual submissions events but navigating the site and other different system actions. It makes sense that the remaining categories only make up a smaller portion of our data.

In [5]:
combined_df["outcome"].value_counts(normalize=True)


outcome
NoFeedback       0.766219
Correct          0.137506
Incorrect        0.067398
Error            0.024980
OtherFeedback    0.003897
Name: proportion, dtype: float64

Drops "cf_instiution" column if it exists which maps each students to an instituion (but everyone goes to michigan)

Creates action_simple column to avoid having a bunch of unique action values.

In [6]:
combined_df = combined_df.drop(columns=["cf_institution"], errors="ignore")

def simplify_action(x):
    if pd.isna(x):
        return "missing"

    value = str(x).strip().lower()
    if value in {"run", "view", "interaction", "viewassignment", "doassignment", "edit"}:
        return value

    return "other"

combined_df["action_simple"] = combined_df["action"].apply(simplify_action)


In [7]:
combined_df.shape

(19805383, 20)

Sorts time for each student and computes current timestamp minus previous timestamp to get time_delta

In [8]:
# Build time_delta (gap between consecutive events per student)
combined_df["time"] = pd.to_datetime(combined_df["time"], errors="coerce")
combined_df = combined_df.sort_values(["anon_student_id", "time"]).copy()

combined_df["time_delta"] = (
    combined_df
    .groupby("anon_student_id", sort=False)["time"]
    .diff()
)

Creates a new column "is_new_session" and "session_index". Each session is defined using a 30 minute inactivity threshold. Session index increases (1-> 2) if time gap exceeds 30 minutes which signal student's 2nd session. is_new_session mark where the new session starts.

In [9]:
session_break = pd.Timedelta(minutes=30)

combined_df["is_new_session"] = (
    combined_df["time_delta"].isna()
    | (combined_df["time_delta"] > session_break)
)

combined_df["session_index"] = (
    combined_df.groupby("anon_student_id")["is_new_session"].cumsum()
)

Allows us to compute how many sessions per student, average session length, attempts per session, if
students who study in shorter sessions perform better, if students return after errors, etc.

Standardize everything to lower case and creates high-level category based on the values in selection to classify the different types of exercises.

Uses vectors for fast column-based logic.

In [10]:
sel = combined_df["selection"].astype("string").str.strip().str.lower()
name = combined_df["problem_name"].astype("string").str.strip().str.lower()

combined_df["exercise_type"] = "Other"

combined_df.loc[sel == "timedexam", "exercise_type"] = "Exam"

combined_df.loc[sel.isin({"activecode", "ac_error", "unittest", "codelens"}), "exercise_type"] = "ActiveCode"

combined_df.loc[sel.isin({"parsonsmove", "parsons", "hparsons", "hparsonsanswer"}), "exercise_type"] = "Parsons"

combined_df.loc[sel.str.contains("dragndrop", na=False), "exercise_type"] = "DragAndDrop"

combined_df.loc[sel.isin({"clickablearea", "selectquestion", "shortanswer", "mchoice", "fillb", "poll"}), "exercise_type"] = "ConceptCheck"

combined_df.loc[(sel == "page") | (name.str.endswith(".html", na=False)), "exercise_type"] = "Reading"

combined_df.loc[sel.isin({
    "assignments", "admin", "dashboard", "default", "group_start", "group_end",
    "peer", "ratepeer", "sendmessage", "video", "audio", "peergroup",
    "view_toggle", "close_toggle", "togglealert", "select_toggle", "designer",
    "endpoint", "practice"
}), "exercise_type"] = "System"

In [11]:
combined_df.exercise_type.value_counts()

exercise_type
ActiveCode      6953614
ConceptCheck    6131407
Parsons         3685430
System          1504761
Reading         1403003
DragAndDrop      110614
Exam              16554
Name: count, dtype: int64

Creates a new dataframe "analysis_df" which only contains rows that belong to a chosen interactive exercise type and that represent actual submission attempts. We removed navigation events, page views, dashboard clicks, .. 

This is important because we saw earlier 76% of our dataset was classified as NoFeedback. Now engagement metrics reflect attempts and not just clicks.

In [12]:
# Normalize once for robust matching
sel = combined_df["selection"].astype("string").str.strip().str.lower()
act = combined_df["action_simple"].astype("string").str.strip().str.lower()
etype = combined_df["exercise_type"]

parsons_submissions = {"parsonsmove", "hparsonsanswer", "hparsons", "parsons"}
conceptcheck_submissions = {"selectquestion", "shortanswer", "mchoice", "fillb", "poll"}

submit_mask = (
    ((etype == "ActiveCode") & act.isin({"run", "interaction"}))
    | ((etype == "Parsons") & sel.isin(parsons_submissions))
    | ((etype == "DragAndDrop") & sel.str.contains("dragndrop", na=False))
    | ((etype == "ConceptCheck") & sel.isin(conceptcheck_submissions))
)

analysis_df = combined_df.loc[
    submit_mask & combined_df["level_subchapter"].notna()
].copy()

Drop any duplicated rows.


In [13]:
analysis_df = analysis_df.drop_duplicates(
    subset=["anon_student_id", "time", "problem_name", "selection", "action", "input"]
)

In [14]:
print("combined_df:", combined_df.shape)
print("analysis_df:", analysis_df.shape)

analysis_df[["exercise_type","action_simple"]].value_counts().head(20)

combined_df: (19805383, 24)
analysis_df: (10502808, 24)


exercise_type  action_simple
ConceptCheck   interaction      4228130
Parsons        other            3684113
ConceptCheck   other            1552561
ActiveCode     run               922983
DragAndDrop    other             110134
ConceptCheck   missing             4832
DragAndDrop    missing               55
Name: count, dtype: int64

We filtered out ~9.3 million rows.
47% of the dataset was removed.
And about 53% remains as actual interactive attempts.

Check to see if we accidentally removed any students:


In [15]:
combined_df["anon_student_id"].nunique()

214

In [16]:
analysis_df["anon_student_id"].nunique()

214

In [17]:
analysis_df["semester"].value_counts()

semester
F25    3281118
F24    1486643
W23    1270235
F23    1195589
F22    1189613
W24     870805
W22     705060
F21     503744
W21          1
Name: count, dtype: int64

W21 has 1 row which is statistically useless. Most likely a test artifcat. We can remove it.

In [18]:
analysis_df = analysis_df[analysis_df["semester"] != "W21"]
combined_df = combined_df[combined_df["semester"] != "W21"]

In [19]:
analysis_df.to_parquet("analysis_df_clean.parquet")